# Keystroke Dynamics Continuous Authentication Pipeline
### Behavior-Based User Verification via Temporal Typing Biometrics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsrashedsatter/thesis-on-keystroke-dynamics-authentication/blob/main/keystroke_dynamics_pipeline.ipynb)

---

## 📌 Executive Summary & Architecture
This notebook trains and evaluates machine learning and deep learning models for continuous behavioral biometric verification using fine-grained keystroke timing data:
- **Cohort**: 60 verified participants with 15 sentence sessions each (**900 sessions** total).
- **Biometrics**: **124 engineered timing features** (Global statistics, Cadence, Errors, and Top 25 Digraph Latencies).
- **Target**: Closed-set multiclass identification across 60 typists (Random Chance baseline = $1/60 \approx 1.67\%$).
- **Evaluated Architectures**:
  1. **Random Forest Classifier** (Baseline ensemble)
  2. **1D Convolutional Neural Network (1D-CNN)** (Local temporal receptive fields)
  3. **Bidirectional LSTM (BiLSTM)** (Recurrent sequence modeling)

## 1. ⚙️ Hardware Diagnostics & Environment Setup

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
print(f"Python Version: {sys.version.split()[0]}")
print(f"Running inside Google Colab: {IN_COLAB}")

# Check for GPU acceleration
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    print(f"TensorFlow Version: {tf.__version__}")
    if gpus:
        print(f"✅ GPU Detected: {gpus}")
    else:
        print("ℹ️ No GPU detected. Running on CPU (CPU is completely fine for this feature matrix).")
        print("   Tip for Colab: Runtime -> Change runtime type -> Select 'T4 GPU' for faster training.")
except ImportError:
    print("TensorFlow will be installed in the next cell.")

## 2. 📥 Clone Repository & Install Dependencies

In [ ]:
if IN_COLAB:
    # Clone repository if not already present
    if not os.path.exists("thesis-on-keystroke-dynamics-authentication"):
        !git clone https://github.com/itsrashedsatter/thesis-on-keystroke-dynamics-authentication.git
        %cd thesis-on-keystroke-dynamics-authentication
    else:
        %cd thesis-on-keystroke-dynamics-authentication
        !git pull origin main

# Install dependencies efficiently
!pip install -q scikit-learn pandas numpy matplotlib seaborn scipy joblib pyarrow tensorflow
print("✅ Dependencies successfully installed and verified.")

## 3. 🔍 Verify Dataset Partitions & Feature Matrix (Stage 03)
Verify the stratified 80/20 train/test partitions generated with zero data leakage (StandardScaler fitted strictly on training partition).

In [ ]:
import pandas as pd
import numpy as np
import json

train_path = "03_feature_engineering_preprocessing/train_scaled.csv"
test_path = "03_feature_engineering_preprocessing/test_scaled.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print(f"Training Partition: {df_train.shape[0]} rows, {df_train.shape[1]} columns")
print(f"Testing Partition:  {df_test.shape[0]} rows, {df_test.shape[1]} columns")
print(f"Target Classes:     {df_train['label'].nunique()} unique participants (0 to 59)")
print(f"Biometric Features: {df_train.shape[1] - 3} features (excluding PARTICIPANT_ID, TEST_SECTION_ID, label)")

df_train.head(3)

## 4. 🌲 Model 1: Random Forest Baseline
Ensemble of 300 decision trees (`n_estimators=300`, `random_state=42`).

In [ ]:
!python 04_model_training/train_random_forest.py

In [ ]:
from IPython.display import Image, display
import json
import pandas as pd

with open("04_model_training/Random_Forest_metrics.json", "r") as f:
    rf_metrics = json.load(f)

rf_acc = rf_metrics.get('test_accuracy', rf_metrics.get('accuracy', 0.0)) * 100
rf_f1_m = rf_metrics.get('macro_f1', rf_metrics.get('f1_macro', 0.0))
rf_f1_w = rf_metrics.get('weighted_f1', rf_metrics.get('f1_weighted', 0.0))

print("=" * 50)
print("🌲 RANDOM FOREST TEST RESULTS:")
print(f"   - Test Accuracy:   {rf_acc:.2f}%")
print(f"   - Macro F1:        {rf_f1_m:.4f}")
print(f"   - Weighted F1:     {rf_f1_w:.4f}")
print("=" * 50)

# Display Top 10 Most Important Biometric Features
rf_fi = pd.read_csv("04_model_training/rf_feature_importances.csv")
print("\nTop 10 Most Discriminative Biometric Features:")
print(rf_fi.head(10).to_string(index=False))

# Confusion Matrix Heatmap
display(Image("04_model_training/Random_Forest_confusion_matrix.png", width=750))

## 5. ⚡ Model 2: 1D Convolutional Neural Network (1D-CNN)
Deep 1D-CNN designed for compact tabular biometric sequences with:
- 2 Convolutional Blocks (`Conv1D` + `BatchNormalization` + `SpatialDropout1D`)
- `GaussianNoise` layer for tabular data augmentation
- `GlobalAveragePooling1D` to prevent parameter explosion
- Stratified internal validation split with early stopping

In [ ]:
!python 04_model_training/train_cnn.py

In [ ]:
from IPython.display import Image, display
import json
import pandas as pd
import matplotlib.pyplot as plt

with open("04_model_training/1D_CNN_metrics.json", "r") as f:
    cnn_metrics = json.load(f)

cnn_acc = cnn_metrics.get('test_accuracy', cnn_metrics.get('accuracy', 0.0)) * 100
cnn_f1_m = cnn_metrics.get('macro_f1', cnn_metrics.get('f1_macro', 0.0))
cnn_f1_w = cnn_metrics.get('weighted_f1', cnn_metrics.get('f1_weighted', 0.0))

print("=" * 50)
print("⚡ 1D-CNN TEST RESULTS:")
print(f"   - Test Accuracy:   {cnn_acc:.2f}%")
print(f"   - Macro F1:        {cnn_f1_m:.4f}")
print(f"   - Weighted F1:     {cnn_f1_w:.4f}")
print("=" * 50)

# Plot Training & Validation Curves
cnn_hist = pd.read_csv("04_model_training/cnn_training_history.csv")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(cnn_hist['loss'], label='Train Loss', color='#2563eb', lw=2)
plt.plot(cnn_hist['val_loss'], label='Val Loss', color='#dc2626', lw=2)
plt.title('1D-CNN: Cross-Entropy Loss vs Epochs', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(cnn_hist['accuracy'], label='Train Accuracy', color='#2563eb', lw=2)
plt.plot(cnn_hist['val_accuracy'], label='Val Accuracy', color='#16a34a', lw=2)
plt.title('1D-CNN: Categorical Accuracy vs Epochs', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

display(Image("04_model_training/1D_CNN_confusion_matrix.png", width=750))

## 6. 🧠 Model 3: Bidirectional LSTM (BiLSTM)
Recurrent architecture leveraging forward and backward sequence transitions across keystroke feature bins:
- `Bidirectional(LSTM(64))` + `Bidirectional(LSTM(32))`
- `BatchNormalization` + `Dropout(0.4)` + $L_2$ Regularization
- Dense classification head with Softmax over 60 participants

In [ ]:
!python 04_model_training/train_lstm.py

In [ ]:
from IPython.display import Image, display
import json
import pandas as pd
import matplotlib.pyplot as plt

with open("04_model_training/LSTM_metrics.json", "r") as f:
    lstm_metrics = json.load(f)

lstm_acc = lstm_metrics.get('test_accuracy', lstm_metrics.get('accuracy', 0.0)) * 100
lstm_f1_m = lstm_metrics.get('macro_f1', lstm_metrics.get('f1_macro', 0.0))
lstm_f1_w = lstm_metrics.get('weighted_f1', lstm_metrics.get('f1_weighted', 0.0))

print("=" * 50)
print("🧠 BiLSTM TEST RESULTS:")
print(f"   - Test Accuracy:   {lstm_acc:.2f}%")
print(f"   - Macro F1:        {lstm_f1_m:.4f}")
print(f"   - Weighted F1:     {lstm_f1_w:.4f}")
print("=" * 50)

# Plot Training & Validation Curves
lstm_hist = pd.read_csv("04_model_training/lstm_training_history.csv")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(lstm_hist['loss'], label='Train Loss', color='#9333ea', lw=2)
plt.plot(lstm_hist['val_loss'], label='Val Loss', color='#dc2626', lw=2)
plt.title('BiLSTM: Cross-Entropy Loss vs Epochs', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(lstm_hist['accuracy'], label='Train Accuracy', color='#9333ea', lw=2)
plt.plot(lstm_hist['val_accuracy'], label='Val Accuracy', color='#16a34a', lw=2)
plt.title('BiLSTM: Categorical Accuracy vs Epochs', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

display(Image("04_model_training/LSTM_confusion_matrix.png", width=750))

## 7. 📊 Consolidated Benchmark & Model Comparison
Comparison of all three architectures evaluated on the same 180 test sessions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

comparison_data = {
    "Model Architecture": ["Random Forest (Baseline)", "Bidirectional LSTM", "1D-CNN"],
    "Test Accuracy (%)": [
        round(rf_acc, 2),
        round(lstm_acc, 2),
        round(cnn_acc, 2)
    ],
    "Macro F1": [
        round(rf_f1_m, 4),
        round(lstm_f1_m, 4),
        round(cnn_f1_m, 4)
    ],
    "Weighted F1": [
        round(rf_f1_w, 4),
        round(lstm_f1_w, 4),
        round(cnn_f1_w, 4)
    ],
    "Zero-Trust Role": [
        "Primary Classifier (Fast & Robust)",
        "Secondary Continuous Temporal Verifier",
        "Compact Deep Learning Candidate"
    ]
}

df_comp = pd.DataFrame(comparison_data)
print("\n--- SUMMARY BENCHMARK TABLE ---")
display(df_comp)

# Visualization of Test Accuracies
plt.figure(figsize=(9, 5))
colors = ['#10b981', '#6366f1', '#f59e0b']
bars = plt.bar(df_comp["Model Architecture"], df_comp["Test Accuracy (%)"], color=colors, width=0.55)
plt.axhline(y=100.0 / 60.0, color='red', linestyle='--', label=f'Random Chance Baseline (1.67%)', alpha=0.7)

plt.ylim(0, 115)
plt.title("Comparative Multiclass Accuracy across 60 Participants", fontsize=13, fontweight='bold')
plt.ylabel("Test Accuracy (%)", fontsize=11)
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.legend(loc='upper right')

for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, h + 2, f"{h:.2f}%", ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

## 8. 🎓 Thesis Methodology & Key Findings

1. **Tree-Based Ensembles Dominate Tabular Biometric Latencies**:
   - Random Forest achieved an outstanding **99.44% Accuracy** and **0.9944 F1**, misclassifying only 1 test session out of 180 across 60 individuals.
   - This confirms that human typing biometrics have distinct, non-linear feature partitioning boundaries that decision trees excel at isolating without requiring millions of training samples.

2. **BiLSTM Successfully Models Temporal Typing Dynamics**:
   - Bidirectional LSTM achieved **86.67% Accuracy**, proving that sequential temporal transitions across digraph timings retain strong subject-specific biometric identity.

3. **CNN Performance on Small Datasets**:
   - 1D-CNN achieved **65.56% Accuracy** (vastly outperforming the 1.67% random guess baseline). While CNNs thrive on massive datasets, our regularized GlobalAveragePooling architecture prevented overfitting on 612 training samples.

4. **Zero-Trust Suitability**:
   - The high accuracy and fast sub-millisecond inference time of Random Forest make it an ideal engine for a real-time background daemon performing continuous identity confidence scoring.